# A2-TempPredict: Temperature Preparation for Simple RNN

This notebook prepares `BASEL_temp_mean` from `Data/weather_prediction_dataset.csv` for a simple PyTorch RNN.

Scenario:
- Feature: `BASEL_temp_mean`
- Sliding window: 14 days
- Forecast horizon: 1 day
- Train/test split: 70% train / 30% test
- 5-fold time-series cross-validation on the training set

## 1. Colab and PyTorch setup

If you run this notebook in Google Colab, either upload the dataset to the notebook workspace or mount Google Drive. The code below works in Colab and local Python notebooks.

In [ ]:
# In Google Colab, uncomment the next line if PyTorch is not installed:
# !pip install torch torchvision torchaudio scikit-learn pandas matplotlib

# If your file is in Google Drive, uncomment these two lines:
# from google.colab import drive
# drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import TimeSeriesSplit
import torch
from torch.utils.data import Dataset, DataLoader
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

## 2. Load dataset and inspect the Basel temperature feature

In [ ]:
data_path = os.path.join('Data', 'weather_prediction_dataset.csv')
df = pd.read_csv(data_path)
df['DATE'] = pd.to_datetime(df['DATE'].astype(str), format='%Y%m%d')
df = df.sort_values('DATE').reset_index(drop=True)

print('Dataset shape:', df.shape)
print('BASEL_temp_mean exists:', 'BASEL_temp_mean' in df.columns)
print(df[['DATE', 'BASEL_temp_mean']].head(10).to_string(index=False))

In [ ]:
missing = df['BASEL_temp_mean'].isna().sum()
print('Missing values:', missing)
print('Value range:', df['BASEL_temp_mean'].min(), 'to', df['BASEL_temp_mean'].max())

## 3. Visualize Basel daily mean temperature

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df['DATE'], df['BASEL_temp_mean'], marker='.', markersize=3, linewidth=1, alpha=0.8)
plt.title('Basel Daily Mean Temperature (BASEL_temp_mean)')
plt.xlabel('Date')
plt.ylabel('Temperature (°C)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Normalize and create sliding windows for RNN input

We use a 14-day lookback and predict one day ahead. The model input shape is `(batch, seq_len, features)` with `features = 1`.

In [ ]:
series = df[['DATE', 'BASEL_temp_mean']].dropna().reset_index(drop=True)
values = series['BASEL_temp_mean'].values.reshape(-1, 1)

scaler = MinMaxScaler(feature_range=(0, 1))
values_scaled = scaler.fit_transform(values).flatten()

print('Rows after dropna:', len(series))
print('Scaled range:', values_scaled.min(), values_scaled.max())

In [ ]:
def create_sliding_windows(data, window_size=14, horizon=1):
    X, y = [], []
    for i in range(len(data) - window_size - horizon + 1):
        X.append(data[i:i + window_size])
        y.append(data[i + window_size + horizon - 1])
    return np.array(X), np.array(y)

window_size = 14
horizon = 1
X, y = create_sliding_windows(values_scaled, window_size=window_size, horizon=horizon)

print('Total sequences:', len(X))
print('X shape:', X.shape)
print('y shape:', y.shape)
print('Example X[0]:', X[0])
print('Example y[0]:', y[0])

## 5. Train/test split (time-order preserved)

Use 70% of the sequences for training and 30% for testing. This split preserves temporal order.

In [ ]:
train_ratio = 0.70
train_size = int(len(X) * train_ratio)

X_train, y_train = X[:train_size], y[:train_size]
X_test, y_test = X[train_size:], y[train_size:]

print('Train samples:', len(X_train))
print('Test samples:', len(X_test))
print('Train ratio:', len(X_train) / len(X))
print('Test ratio:', len(X_test) / len(X))

## 6. 5-fold time-series cross-validation on the training set

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train), start=1):
    print(f'Fold {fold}: train={len(train_idx)}, val={len(val_idx)}')

## 7. Create PyTorch dataset and data loaders

In [ ]:
class SequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(-1)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(-1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = SequenceDataset(X_train, y_train)
test_dataset = SequenceDataset(X_test, y_test)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

batch_x, batch_y = next(iter(train_loader))
print('Train dataset size:', len(train_dataset))
print('Test dataset size:', len(test_dataset))
print('One batch X shape:', batch_x.shape)
print('One batch y shape:', batch_y.shape)

## 8. Simple PyTorch RNN model skeleton

In [ ]:
class SimpleRNN(torch.nn.Module):
    def __init__(self, input_size=1, hidden_size=32, num_layers=1):
        super().__init__()
        self.rnn = torch.nn.RNN(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = torch.nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        out = out[:, -1, :]
        return self.fc(out)

model = SimpleRNN().to(device)
print(model)

## 9. Summary

The Basel temperature dataset is ready for a simple PyTorch RNN model. Continue by adding training/evaluation loops and comparing this scenario with other models.